# Création des PDF

In [1]:
# Cellule 1 — Installation des dépendances Python
# À lancer une seule fois si nécessaire.

%pip install pillow lxml reportlab pikepdf

   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   ---------------------------------------- 7.1/7.1 MB 56.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 80.8 MB/s  0:00:00
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 3.2/3.2 MB 93.9 MB/s  0:00:00

   ---------------------------------------- 0/3 [pillow]
   ------------- -------------------------- 1/3 [reportlab]
   ------------- -------------------------- 1/3 [reportlab]
   ------------- -------------------------- 1/3 [reportlab]
   ------------- -------------------------- 1/3 [reportlab]
   -------------------------- ------------- 2/3 [pikepdf]
   ---------------------------------------- 3/3 [pikepdf]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Cellule 2 — Configuration
from pathlib import Path
import shutil
import subprocess

from lxml import etree
from PIL import Image

from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

import shutil

GS = Path(r"C:\Program Files\gs\gs10.07.1\bin\gswin64c.exe")

# À compléter après installation de veraPDF
VERAPDF = Path(r"C:\Users\rroll\verapdf\verapdf.bat")


DOSSIER_DATA = Path("../data/Frêne_volume_1")

DOSSIER_IMAGES = DOSSIER_DATA / "Images"
DOSSIER_ALTO = DOSSIER_DATA
DOSSIER_SORTIE = DOSSIER_DATA / "exports/pdf"

PDF_INTERMEDIAIRE = DOSSIER_SORTIE / "document_texte_invisible.pdf"
PDF_A2U = DOSSIER_SORTIE / "document_pdfa2u.pdf"

DPI_DEFAUT = 300

NOM_POLICE = "ArialUnicode"

pdfmetrics.registerFont(
    TTFont(NOM_POLICE, "C:/Windows/Fonts/arial.ttf")
)


DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)

In [2]:
print("Ghostscript :", GS.exists())
print("veraPDF :", VERAPDF.exists())

Ghostscript : True
veraPDF : True


In [3]:
# Cellule 3 — Fonctions ALTO

NS_ALTO = {
    "alto": "http://www.loc.gov/standards/alto/ns-v4#"
}

def lire_alto(fichier_alto: Path):
    """Lit un fichier ALTO et retourne l'arbre XML."""
    return etree.parse(str(fichier_alto))


def extraire_dimensions_page(racine):
    """Extrait les dimensions de la page ALTO en pixels."""
    page = racine.find(".//alto:Page", namespaces=NS_ALTO)
    if page is None:
        raise ValueError("Aucune balise <Page> trouvée dans l'ALTO.")

    largeur = float(page.get("WIDTH"))
    hauteur = float(page.get("HEIGHT"))
    return largeur, hauteur


def extraire_lignes_alto(racine):
    """Extrait les lignes et les mots depuis un ALTO."""
    lignes = []

    for ligne in racine.findall(".//alto:TextLine", namespaces=NS_ALTO):
        mots = []
        for string in ligne.findall(".//alto:String", namespaces=NS_ALTO):
            contenu = string.get("CONTENT", "")
            if not contenu.strip():
                continue

            mots.append({
                "texte": contenu,
                "x": float(string.get("HPOS")),
                "y": float(string.get("VPOS")),
                "w": float(string.get("WIDTH")),
                "h": float(string.get("HEIGHT")),
            })

        if mots:
            lignes.append(mots)

    return lignes

In [4]:
# Cellule 4 — Appariement images / ALTO

def trouver_paires_images_alto(dossier_images: Path, dossier_alto: Path):
    """Associe les images et les fichiers ALTO par nom de fichier."""
    extensions = [".tif", ".tiff", ".jpg", ".jpeg", ".png"]
    images = sorted([p for p in dossier_images.iterdir() if p.suffix.lower() in extensions])

    paires = []

    for image in images:
        alto = dossier_alto / f"{image.stem}.xml"

        if alto.exists():
            paires.append((image, alto))
        else:
            print(f"ALTO non trouvé pour : {image.name}")

    return paires


paires = trouver_paires_images_alto(DOSSIER_IMAGES, DOSSIER_ALTO)
print(f"{len(paires)} paire(s) image/ALTO trouvée(s).")

ALTO non trouvé pour : Image00001.tif
ALTO non trouvé pour : Image00002.tif
ALTO non trouvé pour : Image00003.tif
ALTO non trouvé pour : Image00004.tif
ALTO non trouvé pour : Image00005.tif
ALTO non trouvé pour : Image00006.tif
ALTO non trouvé pour : Image00007.tif
ALTO non trouvé pour : Image00009.tif
ALTO non trouvé pour : Image00010.tif
ALTO non trouvé pour : Image00013.tif
ALTO non trouvé pour : Image00014.tif
ALTO non trouvé pour : Image00015.tif
ALTO non trouvé pour : Image00016.tif
ALTO non trouvé pour : Image00020.tif
ALTO non trouvé pour : Image00021.tif
ALTO non trouvé pour : Image00022.tif
ALTO non trouvé pour : Image00023.tif
ALTO non trouvé pour : Image00024.tif
ALTO non trouvé pour : Image00025.tif
ALTO non trouvé pour : Image00026.tif
ALTO non trouvé pour : Image00027.tif
ALTO non trouvé pour : Image00028.tif
ALTO non trouvé pour : Image00029.tif
ALTO non trouvé pour : Image00030.tif
ALTO non trouvé pour : Image00031.tif
ALTO non trouvé pour : Image00032.tif
ALTO non tro

In [5]:
# Cellule 5 — Création du PDF image + texte invisible

def creer_pdf_avec_texte_invisible(paires, fichier_pdf: Path, dpi=DPI_DEFAUT):
    """
    Crée un PDF avec :
    - l'image visible ;
    - le texte ALTO placé en surimpression invisible.
    """

    c = canvas.Canvas(str(fichier_pdf), pageCompression=1)

    for image_path, alto_path in paires:
        image = Image.open(image_path)
        largeur_img_px, hauteur_img_px = image.size

        racine = lire_alto(alto_path)
        largeur_alto_px, hauteur_alto_px = extraire_dimensions_page(racine)
        lignes = extraire_lignes_alto(racine)

        largeur_page_pt = largeur_img_px / dpi * 72
        hauteur_page_pt = hauteur_img_px / dpi * 72

        facteur_x = largeur_page_pt / largeur_alto_px
        facteur_y = hauteur_page_pt / hauteur_alto_px

        c.setPageSize((largeur_page_pt, hauteur_page_pt))

        # Image visible
        c.drawImage(
            str(image_path),
            0,
            0,
            width=largeur_page_pt,
            height=hauteur_page_pt,
            preserveAspectRatio=False,
            mask="auto",
        )

        # Texte invisible
        text_obj = c.beginText()
        text_obj.setTextRenderMode(3)  # 3 = texte invisible dans le PDF
        text_obj.setFont(NOM_POLICE, 10)

        for ligne in lignes:
            for mot in ligne:
                texte = mot["texte"]

                x_pt = mot["x"] * facteur_x
                y_pt = hauteur_page_pt - ((mot["y"] + mot["h"]) * facteur_y)

                taille_police = max(4, mot["h"] * facteur_y * 0.80)

                text_obj.setFont(NOM_POLICE, taille_police)
                text_obj.setTextOrigin(x_pt, y_pt)
                text_obj.textOut(texte + " ")

        c.drawText(text_obj)
        c.showPage()

    c.save()


creer_pdf_avec_texte_invisible(paires, PDF_INTERMEDIAIRE)
print(PDF_INTERMEDIAIRE)

..\data\Frêne_volume_1\exports\pdf\document_texte_invisible.pdf


In [6]:
# Cellule 6 — Test rapide : extraction du texte du PDF

import pikepdf

with pikepdf.open(PDF_INTERMEDIAIRE) as pdf:
    print(f"Nombre de pages : {len(pdf.pages)}")

print("PDF intermédiaire créé.")

Nombre de pages : 6
PDF intermédiaire créé.


In [7]:
# Cellule 7 — Conversion en PDF/A-2u avec Ghostscript

def convertir_pdfa2u_ghostscript(pdf_entree: Path, pdf_sortie: Path):
    """Convertit le PDF vers PDF/A-2u avec Ghostscript."""

    if not GS.exists():
        raise RuntimeError(f"Ghostscript introuvable : {GS}")

    commande = [
        str(GS),
        "-dPDFA=2",
        "-dBATCH",
        "-dNOPAUSE",
        "-dNOOUTERSAVE",
        "-sDEVICE=pdfwrite",
        "-dPDFACompatibilityPolicy=1",
        "-sColorConversionStrategy=RGB",
        "-sProcessColorModel=DeviceRGB",
        "-dEmbedAllFonts=true",
        "-dSubsetFonts=true",
        "-dCompressFonts=true",
        f"-sOutputFile={pdf_sortie}",
        str(pdf_entree),
    ]

    resultat = subprocess.run(
        commande,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    print(resultat.stdout)
    print(resultat.stderr)

    if resultat.returncode != 0:
        raise RuntimeError(
            f"La conversion Ghostscript a échoué.\n\n{resultat.stderr}"
        )

    return pdf_sortie


convertir_pdfa2u_ghostscript(
    PDF_INTERMEDIAIRE,
    PDF_A2U
)

print("PDF/A créé :", PDF_A2U)

GPL Ghostscript 10.07.1 (2026-05-19)
Copyright (C) 2026 Artifex Software, Inc.  All rights reserved.
This software is supplied under the GNU AGPLv3 and comes with NO WARRANTY:
see the file COPYING for details.
Processing pages 1 through 6.
Page 1
Loading font Helvetica (or substitute) from %rom%Resource/Font/NimbusSans-Regular
Page 2
Page 3
Page 4
Page 5
Page 6

The following warnings were encountered at least once while processing this file:
	A CMap has too many code maps.

   **** This file had errors that were repaired or ignored.
   **** The file was produced by: 
   **** >>>> ReportLab PDF Library - (opensource) <<<<
   **** Please notify the author of the software that produced this
   **** file that it does not conform to Adobe's published PDF
   **** specification.



PDF/A créé : ..\data\Frêne_volume_1\exports\pdf\document_pdfa2u.pdf


In [8]:
# Cellule 8 — Validation veraPDF profil PDF/A-2u

def valider_pdfa2u_verapdf(fichier_pdf: Path):
    """Valide le PDF avec veraPDF."""

    if not VERAPDF.exists():
        raise RuntimeError(
            f"veraPDF introuvable : {VERAPDF}"
        )

    commande = [
        str(VERAPDF),
        "--format",
        "text",
        "--flavour",
        "2u",
        str(fichier_pdf),
    ]

    resultat = subprocess.run(
        commande,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    print("=== SORTIE veraPDF ===")
    print(resultat.stdout)

    if resultat.stderr:
        print("=== ERREURS veraPDF ===")
        print(resultat.stderr)

    return resultat.returncode == 0


ok = valider_pdfa2u_verapdf(PDF_A2U)

print(
    "Validation PDF/A-2u :",
    "OK" if ok else "ÉCHEC"
)

=== SORTIE veraPDF ===
FAIL c:\Users\rroll\Documents\GitHub\Projet_Frene\notebook\..\data\Frêne_volume_1\exports\pdf\document_pdfa2u.pdf 2u

Validation PDF/A-2u : ÉCHEC


In [10]:
# Cellule 8 bis — Rapport XML veraPDF

RAPPORT_VERAPDF = DOSSIER_SORTIE / "rapport_verapdf_pdfa2u.xml"

commande = [
    str(VERAPDF),
    "--format",
    "xml",
    "--flavour",
    "2u",
    str(PDF_A2U),
]

resultat = subprocess.run(
    commande,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

RAPPORT_VERAPDF.write_text(resultat.stdout, encoding="utf-8")

print("Rapport veraPDF créé :", RAPPORT_VERAPDF)
print("Validation PDF/A-2u :", "OK" if resultat.returncode == 0 else "ÉCHEC")

Rapport veraPDF créé : ..\data\Frêne_volume_1\exports\pdf\rapport_verapdf_pdfa2u.xml
Validation PDF/A-2u : ÉCHEC
